In [1]:
import pandas as pd

In [2]:
# 1. LOAD CLEANED DATASETS
df_shipments = pd.read_parquet('../01_data/02_interim/clean_shipments.parquet')
df_mapping = pd.read_parquet('../01_data/02_interim/clean_mapping.parquet')
df_coords = pd.read_parquet('../01_data/02_interim/clean_coords.parquet')

In [3]:
# 2. INTEGRATION WORKFLOW

# Execute primary strict merge requiring both division and product context
df_integrated = pd.merge(
    df_shipments,
    df_mapping[['division', 'product_name', 'factory']],
    on=['division', 'product_name'],
    how='left'
)

# Identify and resolve orphaned records bypassing the division constraint
missing_mask = df_integrated['factory'].isnull()

if missing_mask.any():
    print(f"Fixing {missing_mask.sum()} orphans via fallback join...")
    
    # Generate fallback map keyed strictly on product_name
    factory_lookup = df_mapping.set_index('product_name')['factory'].to_dict()
    df_integrated.loc[missing_mask, 'factory'] = df_integrated.loc[missing_mask, 'product_name'].map(factory_lookup)

# Append spatial coordinates to the integrated operational model
df_integrated = pd.merge(
    df_integrated,
    df_coords[['factory', 'latitude', 'longitude']],
    on='factory',
    how='left'
).rename(columns={'latitude': 'factory_lat', 'longitude': 'factory_lon'})

print(f"Orphaned Records after Fix: {df_integrated['factory'].isnull().sum()}")

Fixing 6 orphans via fallback join...
Orphaned Records after Fix: 0


In [4]:
# Export the denormalized base model to the integrated layer
df_integrated.to_parquet('../01_data/03_integrated/logistics_base.parquet', index=False)